**Lecture Explanation: Stacking (Preventing Overfitting)**

"Ab hum baat karte hain **Stacking** ke baare mein aur dekhte hain ki isme **Overfitting** ko kaise prevent kiya jata hai.

Stacking ka basic architecture hume pata hai:

1. Hamare paas ek **Dataset** hota hai jise hum alag-alag base models ko dete hain—jaise **Model 1**, **Model 2**, aur **Model 3**.
2. Ye teeno models train hokar apne predictions dete hain: $P_1$, $P_2$, aur $P_3$.
3. In teeno predictions ko hum as input features ek **Meta-Model** ko pass karte hain, jo aage process karke hamara **Final Prediction** nikaalta hai.

**Overfitting ki problem kyu aati hai?**
Agar hum base models ko train karne ke baad usi same data par predictions nikaal kar direct meta-model ko de denge, toh meta-model data ko memorize kar lega aur **Data Leakage / Overfitting** ho jayegi.

Is problem ko solve karne ke liye 2 methods use hote hain:

1. **K-Fold CV / K-Fold Method:**
* Hum apne dataset ko $K$ equal folds me divide karte hain (jaise yahan humne $K = 3$ liya hai: Fold 1, Fold 2, Fold 3).
* **Iteration 1:** Fold 1 aur Fold 2 par base models ko train karenge, aur bache hue Fold 3 ko as **Validation Data** rakh kar uspar predictions generate karenge.
* **Iteration 2:** Fold 1 aur Fold 3 par train karenge aur Fold 2 par validation predictions lenge.
* **Iteration 3:** Fold 2 aur Fold 3 par train karke Fold 1 par predict karenge.
* Is tarah poore dataset ke liye unseen data ke out-of-fold predictions ($P_1, P_2, P_3$) mil jaate hain, jinse hum meta-model ko bina overfitting ke safely train karte hain.


2. **Blending:**
* Iska doosra tarika hota hai Blending, jisme ek separate hold-out validation set banakar meta-model train kiya jata hai."

**Lecture Explanation: Blending (Preventing Overfitting)**

"Ab hum samajhte hain Stacking me overfitting rokne ka doosra method—**Blending**.

K-Fold CV me baar-baar alag-alag folds par models train ho rahe the, jisme time aur computation zyada lagta hai. Blending ek simpler aur faster approach hai.

**Blending ka step-by-step workflow:**

1. **Dataset Split:** Sabse pehle poore dataset ko 3 parts me divide karte hain:
* **Train Set**
* **Validation Set** (Hold-out set)
* **Test Set**


2. **Base Models Training:**
* Hamare base models (**Model 1**, **Model 2**, **Model 3**) sirf aur sirf **Train Set** par train hote hain.


3. **Meta-Set Generation (Validation Phase):**
* Train hone ke baad, yeh teeno base models **Validation Set** par predictions generate karte hain: `val_pred1`, `val_pred2`, aur `val_pred3`.
* In teeno predictions aur validation set ke actual labels ko jodkar ek naya dataset banta hai jise **Meta Set** (Validation Dataset) kehte hain.


4. **Meta-Model Training:**
* Is **Meta-Set** par hum apne **Meta-Model** ko train karte hain. Kyunki base models ne validation set ko training me nahi dekha tha, isliye yahan koi data leakage ya overfitting nahi hoti.


5. **Final Testing Phase:**
* **Test Set** ko pehle base models ko diya jata hai, jisse milte hain `test_pred1`, `test_pred2`, `test_pred3`.
* Yeh test predictions Meta-Model ko pass hoti hain, aur Meta-Model hamara **Final Prediction / Test Evaluation** nikaalta hai."



---

**Asaan Bhasha Me: Validation Data vs Test Data me kya difference hai?**

Is pure concept ko ek **Exam Analogy** se samjho:

* **1. Train Set (Syllabus/Textbook):**
* Yeh wo data hai jisse base models padhai karte hain aur concepts seekhte hain.


* **2. Validation Set (Mock Test / Pre-Board):**
* Yeh intermediate test hai. Iska use base models ki kamzori samajhne ke liye hota hai taaki Meta-Model yeh seekh sake ki *'kaunsa base model kab sahi bolta hai aur kab galat'*. Is test ke feedback se Meta-Model train hota hai.


* **3. Test Set (Final Board Exam):**
* Yeh wo unseen exam hai jo pehle kabhi kisi model ne nahi dekha. Iska use sirf aur sirf final system ki actual performance check karne ke liye hota hai. Is par koi learning ya adjustment nahi hoti.



---

**Quick Comparison**

| Feature | Validation Data | Test Data |
| --- | --- | --- |
| **Main Kaam** | Meta-Model ko train karna aur model tuning karna | Pure pipeline ki final accuracy measure karna |
| **Model Par Asar** | Iske predictions se Meta-Model seekhta hai (indirect training) | Isse koi model kuch nahi seekhta (pure evaluation) |
| **Kab Use Hota Hai?** | Development / Training phase ke dauran | Bilkul aakhri step me, jab pura system ready ho |

In [3]:
# 1. Imports
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import StackingRegressor
from sklearn.metrics import r2_score

# 2. Dataset Generation & Splitting
# (Notebook me regression dataset use kiya gaya hai)
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [4]:

# 3. Base Models (Level-0 Estimators)
lin_reg = LinearRegression()
dtr = DecisionTreeRegressor(max_depth=3)
svr = SVR()

# Estimators list define karna
estimators = [
    ('lr', lin_reg),
    ('dtr', dtr),
    ('svr', svr)
]

# 4. Stacking Regressor (Level-1 Meta-Model)
sr = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV()
)

# 5. Model Training
sr.fit(X_train, y_train)

# 6. Predictions
y_pred = sr.predict(X_test)
y_pred_train = sr.predict(X_train)

# 7. Evaluation
print("r2 score test:", r2_score(y_test, y_pred))
print("r2 score train:", r2_score(y_train, y_pred_train))

r2 score test: 0.9999994341163647
r2 score train: 0.999999473375613
